In [ ]:
%pip install torch datasets tiktoken tqdm scikit-learn

Traceback (most recent call last):
  File "c:\Users\zaack\AppData\Local\Python\bin\pip.exe.__script__.py", line 33, in <module>
    from pip._internal.cli.main import main
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1138, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1078, in _find_spec
  File "C:\Users\zaack\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\_distutils_hack\__init__.py", line 109, in find_spec
    return method()
           ^^^^^^^^
  File "C:\Users\zaack\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\_distutils_hack\__init__.py", line 157, in spec_for_pip
    if sys.version_info >= (3, 12) or self.pip_imported_during_build():
                                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\zaack\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\_distutils_hack\__init__.py", line 169, in pip_imported_during_build
    return a

In [2]:
import math
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tqdm import tqdm
import tiktoken

from sklearn.metrics import classification_report, confusion_matrix

C:\Users\zaack\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [4]:
dataset = load_dataset("stanfordnlp/imdb")

train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

len(train_texts), len(test_texts)

(25000, 25000)

In [5]:
tokenizer = tiktoken.get_encoding("gpt2")
base_vocab_size = tokenizer.n_vocab
base_vocab_size

50257

In [6]:
PAD_ID = base_vocab_size
CLS_ID = base_vocab_size + 1
SEP_ID = base_vocab_size + 2

VOCAB_SIZE = base_vocab_size + 3
VOCAB_SIZE



50260

In [7]:
import random

_previous_colors = []

def cprint(*args):
    global _previous_colors

    while True:
        # Generate a random background color
        bg = tuple(random.randint(50, 230) for _ in range(3))

        # Calculate perceived luminance
        r, g, b = bg
        luminance = (0.2126 * r + 0.7152 * g + 0.0722 * b)

        # Pick a readable foreground
        fg = (0, 0, 0) if luminance > 140 else (255, 255, 255)

        color = (fg, bg)

        if color not in _previous_colors:
            break

    _previous_colors.append(color)

    if len(_previous_colors) > 2:
        _previous_colors.pop(0)

    fr, fg, fb = color[0]
    br, bg, bb = color[1]

    print(
        f"\033[38;2;{fr};{fg};{fb}m"
        f"\033[48;2;{br};{bg};{bb}m",
        *args,
        "\033[0m",
        sep="",
        end=""
    )

def tokenize(text):    
    tokens = tokenizer.encode(text)

    for token in tokens:
        if not tokenizer.decode([token]).strip():
            cprint(f"[{token}]{tokenizer.decode([token])}")
        else:
            cprint(tokenizer.decode([token]))

In [8]:
text = '''The morning sun drifted across the quiet forest, painting 
the tall pines in warm shades of gold and amber. A soft breeze 
whispered through the leaves, carrying the fresh scent of damp 
earth and wild pine needles. Down by the winding river, a small red 
fox stepped cautiously onto the smooth stones. It paused, tilted its 
head, and listened to the gentle rush of the clear water below. For 
a brief moment today, the entire woods stood completely still, wrapped 
in a peaceful morning silence before the busy rhythm of the day truly 
began to wake up the sleeping valley.'''

tokenize(text)

The morning sun drifted across the quiet forest, painting[220] [198]
the tall pines in warm shades of gold and amber. A soft breeze[220] [198]
whispered through the leaves, carrying the fresh scent of damp[220] [198]
earth and wild pine needles. Down by the winding river, a small red[220] [198]
fox stepped cautiously onto the smooth stones. It paused, tilted its[220] [198]
head, and listened to the gentle rush of the clear water below. For[220] [198]
a brief moment today, the entire woods stood completely still, wrapped[220] [198]
in a peaceful morning silence before the busy rhythm of the day truly[220] [198]
began to wake up the sleeping valley.

In [9]:
MAX_LEN = 256

def encode(text):
    token_ids = tokenizer.encode(text)
    token_ids = token_ids[:MAX_LEN - 2]

    token_ids = [CLS_ID] + token_ids + [SEP_ID]

    if len(token_ids) < MAX_LEN:
        token_ids += [PAD_ID] * (MAX_LEN - len(token_ids))

    return token_ids

In [10]:
def create_attention_mask(input_ids):
    return (input_ids != PAD_ID).long()

In [11]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = torch.tensor(encode(self.texts[idx]))
        mask = create_attention_mask(ids)
        label = torch.tensor(self.labels[idx])

        return ids, mask, label

In [12]:
train_ds = IMDBDataset(train_texts, train_labels)
test_ds = IMDBDataset(test_texts, test_labels)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16)

In [13]:
class BERTEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_len, dropout=0.1):
        super().__init__()
        self.token = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.position = nn.Embedding(max_len, embed_dim)
        self.segment = nn.Embedding(2, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T = x.size()
        pos = torch.arange(T).unsqueeze(0).to(x.device)
        seg = torch.zeros_like(x)

        embeddings = (
            self.token(x) +
            self.position(pos) +
            self.segment(seg)
        )

        embeddings = self.norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

In [14]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, heads, dropout=0.1):
        super().__init__()
        assert dim % heads == 0

        self.heads = heads
        self.d = dim // heads

        self.qkv = nn.Linear(dim, dim * 3)
        self.out = nn.Linear(dim, dim)

        self.attn_dropout = nn.Dropout(dropout)
        self.out_dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        B, T, C = x.shape

        q, k, v = self.qkv(x).chunk(3, dim=-1)

        q = q.view(B, T, self.heads, self.d).transpose(1, 2)
        k = k.view(B, T, self.heads, self.d).transpose(1, 2)
        v = v.view(B, T, self.heads, self.d).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d)

        mask = mask.unsqueeze(1).unsqueeze(2)
        scores = scores.masked_fill(mask == 0, -1e9)

        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)

        out = attn @ v
        out = out.transpose(1, 2).reshape(B, T, C)

        out = self.out(out)
        out = self.out_dropout(out)
        return out

In [15]:
class FeedForward(nn.Module):
    def __init__(self, dim, hidden, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [16]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(dim, heads, dropout)
        self.ff = FeedForward(dim, ff_dim, dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x, mask):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ff(self.norm2(x))
        return x

In [17]:
class BERTEncoder(nn.Module):
    def __init__(self, vocab_size, dim, max_len, layers, heads, ff_dim):
        super().__init__()
        self.embed = BERTEmbedding(vocab_size, dim, max_len)
        self.layers = nn.ModuleList([
            TransformerBlock(dim, heads, ff_dim)
            for _ in range(layers)
        ])

    def forward(self, x, mask):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x, mask)
        return x

In [18]:
class BERTForClassification(nn.Module):
    def __init__(self, vocab_size, dim, max_len, layers, heads, ff_dim):
        super().__init__()
        self.bert = BERTEncoder(
            vocab_size, dim, max_len, layers, heads, ff_dim
        )
        self.classifier = nn.Sequential(nn.Dropout(0.1),nn.Linear(dim, 2))

    def forward(self, x, mask):
        out = self.bert(x, mask)
        cls = out[:, 0]
        return self.classifier(cls)

In [19]:
model = BERTForClassification(
    vocab_size=VOCAB_SIZE,
    dim=256,
    max_len=MAX_LEN,
    layers=4,
    heads=4,
    ff_dim=1024
).to(device)

print(model)

BERTForClassification(
  (bert): BERTEncoder(
    (embed): BERTEmbedding(
      (token): Embedding(50260, 256, padding_idx=50257)
      (position): Embedding(256, 256)
      (segment): Embedding(2, 256)
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layers): ModuleList(
      (0-3): 4 x TransformerBlock(
        (attn): MultiHeadSelfAttention(
          (qkv): Linear(in_features=256, out_features=768, bias=True)
          (out): Linear(in_features=256, out_features=256, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (out_dropout): Dropout(p=0.1, inplace=False)
        )
        (ff): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=256, out_features=1024, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.1, inplace=False)
            (3): Linear(in_features=1024, out_features=256, bias=True)
            (4): Dropout(p=

In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

In [21]:
EPOCHS = 100

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for ids, mask, labels in tqdm(train_loader):
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {total_loss:.2f}")

100%|██████████| 1563/1563 [02:20<00:00, 11.11it/s]


Epoch 1 | Train Loss: 1020.50


100%|██████████| 1563/1563 [02:21<00:00, 11.08it/s]


Epoch 2 | Train Loss: 821.74


100%|██████████| 1563/1563 [02:14<00:00, 11.65it/s]


Epoch 3 | Train Loss: 727.62


... (52 outputs left out)

100%|██████████| 1563/1563 [02:56<00:00,  8.83it/s]


Epoch 30 | Train Loss: 50.23


 66%|██████▋   | 1038/1563 [01:54<00:58,  9.04it/s]


KeyboardInterrupt: 

In [24]:
all_preds, all_labels = [], []

model.eval()
with torch.no_grad():
    for ids, mask, labels in test_loader:
        preds = model(ids.to(device), mask.to(device)).argmax(dim=1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=["Negative", "Positive"]))

              precision    recall  f1-score   support

    Negative       0.78      0.88      0.82     12500
    Positive       0.86      0.75      0.80     12500

    accuracy                           0.81     25000
   macro avg       0.82      0.81      0.81     25000
weighted avg       0.82      0.81      0.81     25000



In [23]:
SAVE_DIR = "bert_from_scratch_imdb"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(model.state_dict(), f"{SAVE_DIR}/model.pt")

torch.save({
    "pad_id": PAD_ID,
    "cls_id": CLS_ID,
    "sep_id": SEP_ID,
    "max_len": MAX_LEN
}, f"{SAVE_DIR}/tokenizer_info.pt")

print("Model saved successfully!")

Model saved successfully!


In [25]:
model.load_state_dict(torch.load(f"{SAVE_DIR}/model.pt", map_location=device))
model.eval()
print("Model loaded successfully!")

Model loaded successfully!


In [42]:
ratings = [
    "I truly hated this film. For what it is worth, it did make me think a bit. But the thinking was pointless, it tried to be meta and deep but had no wisdom to impart."
   ,"It was wonderful. Even though it had hiccups, overall it managed to captivate me till the end."
   ,"The movie was a whole bunch of boring nothingness. I did enjoy the cinematography though."
   ,"I'm honestly not sure what to think about it. It had its weak and good parts. But it had conflicting ideals overall."
]

for text in ratings:
    print("Understood input:")
    tokenize(text)
    print()

    ids = torch.tensor([encode(text)]).to(device)
    mask = (ids != PAD_ID).long()

    with torch.no_grad():
        pred = model(ids, mask).argmax(dim=1).item()

    print("Prediction:", "Positive\n" if pred == 1 else "Negative\n")

Understood input:
I truly hated this film. For what it is worth, it did make me think a bit. But the thinking was pointless, it tried to be meta and deep but had no wisdom to impart.
Prediction: Negative

Understood input:
It was wonderful. Even though it had hiccups, overall it managed to captivate me till the end.
Prediction: Positive

Understood input:
The movie was a whole bunch of boring nothingness. I did enjoy the cinematography though.
Prediction: Negative

Understood input:
I'm honestly not sure what to think about it. It had its weak and good parts. But it had conflicting ideals overall.
Prediction: Negative

